In [ ]:
# We create contact class (OOP) 

class Contact:
    def __init__(self, name, job_title, phone="", email=""):
        self.name = name.strip()
        self.job_title = job_title.strip()
        self.phone = phone.strip()
        self.email = email.strip()

    def __str__(self):
        return f"{self.name} | {self.job_title} |{self.phone} | {self.email}"

    def to_dict(self):
        return {"name": self.name, "job_title": self.job_title, "phone": self.phone, "email": self.email}

    @staticmethod
    def from_dict(data):
        return Contact(data.get("name", ""), data.get("job_title", ""), data.get("phone", ""), data.get("email", ""))

In [ ]:
# --- Help compatibility dict <-> Contact ---

def get_name(c):
    return c.name if isinstance(c, Contact) else c["name"]

def get_job_title(c):
    return c.job_title if isinstance(c, Contact) else c["job_title"]

def get_phone(c):
    return c.phone if isinstance(c, Contact) else c.get("phone","")

def get_email(c):
    return c.email if isinstance(c, Contact) else c.get("email","")

In [ ]:
# Step 1 : data input + basic printing

contacts = [{"name": "Alice Smith", "job_title": "Finance Director", "phone": "+987654321", "email": "alice.smith@email.com"},
    {"name": "John Doe", "job_title": "HR Director", "phone": "+123456789", "email": "john.doe@email.com"},]

def view_contacts(contacts):
    if not contacts:
        info("No contacts found")
        return
    print("==== All Contacts ====")
    for i, c in enumerate(contacts, 1):
        print(f"{i}. {get_name(c)} | {get_job_title(c)} | {get_phone(c)} | {get_email(c)}")
    print("======================")
    print(f"Total: {len(contacts)} contacts")

def save_contacts(contacts):
    import json
    with open("contacts.json", "w") as f:
        json.dump([c.to_dict() for c in contacts], f)

view_contacts(contacts)
contacts = [Contact.from_dict(c) if isinstance(c, dict) else c for c in contacts]
save_contacts(contacts)

In [ ]:
# Install colorama for colors
# pip install colorama

from colorama import Fore, Style, init
init(autoreset=True)

def info(msg):
    print(f"{Fore.CYAN}[INFO]{Style.RESET_ALL} {msg}")

def success(msg):
    print(f"{Fore.GREEN}[SUCCESS]{Style.RESET_ALL} {msg}")

def warning(msg):
    print(f"{Fore.YELLOW}[WARNING]{Style.RESET_ALL} {msg}")

def error(msg):
    print(f"{Fore.RED}[ERROR]{Style.RESET_ALL} {msg}")


def add_contact(contacts):
    name = input("Enter full name: ")
    job_title = input("Enter job title: ")
    phone = input("Enter phone number: ")
    email = input("Enter email address: ")

    if not name or not job_title or not phone or not email:
        warning("All fields are required!")
        return

    if not is_valid_phone(phone):
        error("Invalid phone number format! Example: +33612345678 or 0612345678")
        return

    for c in contacts:
        if c.name.lower() == name.lower():
            warning(f"A contact named '{name}' already exists.")
            return

    contact = Contact(name, job_title, phone, email)
    contacts.append(contact)
    save_contacts(contacts)
    success(f"Contact '{name}' added successfully!")

def view_contacts(contacts):
    if not contacts:
        warning("No contacts found.")
        return

    print(Fore.CYAN + "==== All Contacts ====" + Style.RESET_ALL)

    for i, c in enumerate(contacts, 1):
        print(f"{i}. {c.name} | {getattr(c, 'job_title', '')} | {getattr(c, 'phone', '')} | {getattr(c, 'email', '')}")

    info(f"Total: {len(contacts)} contacts")

In [ ]:
# Step 2 : Save and upload in JSON

import json, os

# in order to put Color-code status messages using colorama
try:
    from colorama import Fore, Style, init
    init(autoreset=True)

    def ok(msg):   print(Fore.GREEN  + msg)        # success
    def info(msg): print(Fore.CYAN   + msg)        # neutral information
    def warn(msg): print(Fore.YELLOW + msg)        # warning
    def err(msg):  print(Fore.RED    + msg)        # error
except ImportError:
    def ok(msg):   print(msg)
    def info(msg): print(msg)
    def warn(msg): print(msg)
    def err(msg):  print(msg)

DATA_FILE = "contacts_tutor.json"

def save_contacts(contacts, path=DATA_FILE):
    with open(path, "w", encoding="utf-8") as f:
        json.dump([c.to_dict() for c in contacts], f, ensure_ascii=False, indent=2)

def load_contacts(path=DATA_FILE):
    if not os.path.exists(path):
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return [Contact.from_dict(c) for c in data]
    except (json.JSONDecodeError, OSError):
        return []

contacts = [Contact.from_dict(c) if isinstance(c, dict) else c for c in contacts]
save_contacts(contacts)

save_contacts(contacts)         
reloaded = load_contacts()     
view_contacts(reloaded)

In [ ]:
# Step 3 : Add a contact (with little verification)

import re

def find_contact_index(contacts, name):
    name_lower = name.strip().lower()
    for i, c in enumerate(contacts):
        if get_name(c).strip().lower() == name_lower:
            return i
    return None

def is_valid_email(email: str) -> bool:
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email))

def is_valid_phone(phone: str) -> bool:
    phone = phone.strip()
    if not re.match(r"^\+?[0-9\s\-\(\)]+$", phone):
        return False
    digits_only = re.sub(r"\D", "", phone)
    return 8 <= len(digits_only) <= 15

def add_contact_interactive(contacts):
    name  = input("Enter full name: ").strip()
    if not name:
        err("Name cannot be empty.")
        return

    if find_contact_index(contacts, name) is not None:
        err(f'Contact "{name}" already exists.')
        return

    job_title = input("Enter job title: ").strip()
    phone = input("Enter phone number: ").strip()
    email = input("Enter email address: ").strip()

    if phone and not is_valid_phone(phone):
        err("Invalid phone format.")
        return
    if email and not is_valid_email(email):
        err("Invalid email format.")
        return

    contacts.append(Contact(name=name, job_title=job_title, phone=phone, email=email))
    ok(f'Contact "{name}" added successfully!')

contacts_mem = load_contacts()
view_contacts(contacts_mem)
add_contact_interactive(contacts_mem)
view_contacts(contacts_mem)
save_contacts(contacts_mem)

In [ ]:
# Step 4 : Search and edit
# Goal : filter by name, edit a contact

def search_contact(contacts, query):
    q = query.strip().lower()
    return [c for c in contacts if q in get_name(c).lower()]

def edit_contact_interactive(contacts):
    name = input("Enter name of contact to edit: ").strip()
    idx = find_contact_index(contacts, name)
    if idx is None:
        err("Contact not found.")
        return

    current = contacts[idx]

    print("Current contact info:")
    print(f"Name: {get_name(current)}")
    print(f"Job title: {get_job_title(current)}")
    print(f"Phone: {get_phone(current)}")
    print(f"Email: {get_email(current)}")

    new_name  = input("Enter new name (leave blank to keep current): ").strip() or get_name(current)
    new_job_title  = input("Enter new job title (leave blank to keep current): ").strip() or get_job_title(current)
    new_phone = input("Enter new phone (leave blank to keep current): ").strip() or get_phone(current)
    new_email = input("Enter new email (leave blank to keep current): ").strip() or get_email(current)

    if new_phone and not is_valid_phone(new_phone):
        err("Invalid phone format.")
        return
    if new_email and not is_valid_email(new_email):
        err("Invalid email format.")
        return

    if new_name.strip().lower() != get_name(current).strip().lower():
        if find_contact_index(contacts, new_name) is not None:
            err(f'Another contact with the name "{new_name}" already exists.')
            return

    contacts[idx] = Contact(name=new_name.strip(), job_title=new_job_title.strip(), phone=new_phone.strip(), email=new_email.strip())
    ok("Contact updated!")
  

contacts_mem = load_contacts()
edit_contact_interactive(contacts_mem)
save_contacts(contacts_mem)

In [ ]:
# Step 5 : Delete and sort
# Goal : delete a contact + systematic sorted printing

def delete_contact_interactive(contacts):
    name = input("Enter name of contact to delete: ").strip()
    idx = find_contact_index(contacts, name)
    if idx is None:
        err("Contact not found.")
        return
    confirm = input(f'Are you sure you want to delete "{get_name(contacts[idx])}"? (yes/no): ').strip().lower()
    if confirm in ("yes", "y", "oui", "o", "YES", "OUI"):
        contacts.pop(idx)
        warn("Contact deleted!")
    else:
        info("Cancelled.")

def view_contacts_sorted(contacts):
    if not contacts:
        info("No contacts.")
        return
    sorted_contacts = sorted(contacts, key=lambda c: get_name(c).lower())
    print("==== All Contacts (sorted) ====")
    for i, c in enumerate(sorted_contacts, 1):
        print(f"{i}. {get_name(c)} | {get_job_title(c)} | {get_phone(c)} | {get_email(c)}")
    print("===============================")
    print(f"Total: {len(sorted_contacts)}")

contacts_mem = load_contacts()
view_contacts_sorted(contacts_mem)
delete_contact_interactive(contacts_mem)
view_contacts_sorted(contacts_mem)
save_contacts(contacts_mem)

In [ ]:
# Export in csv format

import csv

def export_contacts_to_csv(contacts, filename="contacts_export.csv"):
    rows = [c.to_dict() if isinstance(c, Contact) else c for c in contacts]
    fieldnames = ["name", "phone", "email", "job_title"]

    try:
        with open(filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=fieldnames,
                delimiter=',',
                extrasaction="ignore"  
            )
            writer.writeheader()
            writer.writerows(rows)
        success(f"Contacts exported successfully in: {filename}")
    except Exception as e:
        error(f"Error during the csv export: {e}")

In [ ]:
# Password creation

PASSWORD = "Mr Planas-Bielsa is the best teacher ever"  

def require_password():
    print("==== Protected Contact Book ====")
    for attempt in range(3):
        pwd = input("Enter password: ").strip()
        if pwd == PASSWORD:
            print("Access granted.\n")
            return
        else:
            print("Wrong password.")
    print("Too many failed attempts. Exiting.")
    exit()

In [ ]:
# Statistics about this contact book

from collections import Counter

def get_field(contact, field: str) -> str:
    """Retrieve a field value for a contact stored as a dict OR as a Contact object."""
    if isinstance(contact, dict):
        return (contact.get(field) or "").strip()
    return (getattr(contact, field, "") or "").strip()


def show_statistics(contacts):
    print("==== Statistics ====")
    if not contacts:
        print("No contacts to analyze.")
        return

    total = len(contacts)
    no_phone = 0
    no_email = 0
    domains = Counter()
    job_titles = Counter()

    for c in contacts:
        phone = get_field(c, "phone")
        email = get_field(c, "email")
        job = get_field(c, "job_title")

        if not phone:
            no_phone += 1
        if not email:
            no_email += 1

        if "@" in email:
            domain = email.split("@")[-1].lower()
            domains[domain] += 1

        if job:
            job_titles[job] += 1

    print(f"Total contacts: {total}")
    print(f"Contacts without phone: {no_phone}")
    print(f"Contacts without email: {no_email}")

    if domains:
        print("\nTop email domains:")
        for domain, count in domains.most_common():
            print(f"- {domain}: {count} contact(s)")

    if job_titles:
        print("\nJob title distribution:")
        for job, count in job_titles.most_common():
            print(f"- {job}: {count} contact(s)")

    print("====================")

In [ ]:
# Step 6 : Interface launchinf (mini-interface CLI)

def main_loop_tutor():
    require_password()
    contacts = load_contacts()
    while True:
        print("===============================")
        print("Contact Book")
        print("===============================")
        print("1. Add Contact")
        print("2. View All Contacts")
        print("3. Search Contact by Name")
        print("4. Edit Contact")
        print("5. Delete Contact")
        print("6. Save and Exit")
        print("7. Export to CSV")
        print("8. Show Statistics")
        choice = input("Choose (1-8): ").strip()

        if choice == "1":
            add_contact_interactive(contacts)
        elif choice == "2":
            view_contacts_sorted(contacts)
        elif choice == "3":
            q = input("Enter name to search: ").strip()
            results = search_contact(contacts, q)
            if results:
                print(f"Found {len(results)} result{'s' if len(results) != 1 else ''}:")
                for i, c in enumerate(results, 1):
                    print(f"{i}. {c.name} | {getattr(c, 'job_title', '')} | {getattr(c, 'phone', '')} | {getattr(c, 'email', '')}")
            else:
                warning(f'No contact found with name containing "{q}".')
        elif choice == "4":
            edit_contact_interactive(contacts)
        elif choice == "5":
            delete_contact_interactive(contacts)
        elif choice == "6":
            info("Saving contacts to file...")
            save_contacts(contacts)
            ok("Data saved. Goodbye!")
            break
        elif choice == "7":
            export_contacts_to_csv(contacts)
        elif choice == "8":
            show_statistics(contacts)
        else:
            err("Invalid choice.")

main_loop_tutor()